# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 7: Interactive Dashboard with Plotly Dash

This notebook runs the dashboard from `7_spacex_dash_app.py` inline. Run both cells; the dashboard appears under the last one.

In [1]:
%pip install -q dash plotly pandas

In [2]:
# SpaceX Launch Records Dashboard (Plotly Dash)
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

# Read the SpaceX launch data into a pandas dataframe
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv")
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

launch_sites = sorted(spacex_df['Launch Site'].unique())

# Create a dash application
app = dash.Dash(__name__)
app.title = "SpaceX Launch Records Dashboard"

# Create the app layout
app.layout = html.Div(children=[
    html.H1('SpaceX Launch Records Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),

    # TASK 1: Dropdown list to select a launch site (default: all sites)
    dcc.Dropdown(id='site-dropdown',
                 options=[{'label': 'All Sites', 'value': 'ALL'}] +
                         [{'label': site, 'value': site} for site in launch_sites],
                 value='ALL',
                 placeholder="Select a Launch Site here",
                 searchable=True),
    html.Br(),

    # TASK 2: Pie chart of successful launches
    # All sites: share of successful launches by site. One site: success vs failure counts.
    html.Div(dcc.Graph(id='success-pie-chart')),
    html.Br(),

    html.P("Payload range (Kg):"),
    # TASK 3: Slider to select the payload range
    dcc.RangeSlider(id='payload-slider',
                    min=0, max=10000, step=1000,
                    marks={0: '0', 2500: '2500', 5000: '5000', 7500: '7500', 10000: '10000'},
                    value=[min_payload, max_payload]),

    # TASK 4: Scatter chart of payload vs launch outcome
    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
])


# TASK 2: Callback: site-dropdown (input) -> success-pie-chart (output)
@app.callback(Output(component_id='success-pie-chart', component_property='figure'),
              Input(component_id='site-dropdown', component_property='value'))
def get_pie_chart(entered_site):
    if entered_site == 'ALL' or entered_site is None:
        fig = px.pie(spacex_df, values='class', names='Launch Site',
                     title='Total Successful Launches by Site')
    else:
        filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]
        counts = filtered_df.groupby('class').size().reset_index(name='count')
        counts['Outcome'] = counts['class'].map({1: 'Success', 0: 'Failure'})
        fig = px.pie(counts, values='count', names='Outcome',
                     color='Outcome', color_discrete_map={'Success': 'green', 'Failure': 'red'},
                     title='Launch Success vs Failure for site {}'.format(entered_site))
    return fig


# TASK 4: Callback: site-dropdown and payload-slider (inputs) -> success-payload-scatter-chart (output)
@app.callback(Output(component_id='success-payload-scatter-chart', component_property='figure'),
              [Input(component_id='site-dropdown', component_property='value'),
               Input(component_id='payload-slider', component_property='value')])
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range
    filtered_df = spacex_df[(spacex_df['Payload Mass (kg)'] >= low) &
                            (spacex_df['Payload Mass (kg)'] <= high)]
    if entered_site == 'ALL' or entered_site is None:
        title = 'Correlation between Payload and Success for all Sites'
    else:
        filtered_df = filtered_df[filtered_df['Launch Site'] == entered_site]
        title = 'Correlation between Payload and Success for site {}'.format(entered_site)
    fig = px.scatter(filtered_df, x='Payload Mass (kg)', y='class',
                     color='Booster Version Category', title=title)
    return fig


# Run the app inside the notebook (Colab / Jupyter)
app.run(jupyter_mode='inline', jupyter_height=1000, port=8051)

<IPython.core.display.Javascript object>